# Recommender Model 1: Nearest-Neighbor (Euclidean Distance) Recommender

**Python port of `Recommender_model_1.R`** — same logic (scale musical features, rank songs by Euclidean-norm distance, boost shared-artist matches), no R required.

Given a song, recommend the 5 nearest songs in scaled musical-feature space, boosting songs that share an artist with the query song to the top of the list.

In [1]:
import pandas as pd
import numpy as np
import difflib
from sklearn.preprocessing import StandardScaler
import os

In [2]:
DATA_DIR = 'data'
data = pd.read_csv(os.path.join(DATA_DIR, 'spotify-2023.csv'), encoding='latin-1')

## Prepare Feature Matrix

Select the musical features (matching the R original's `bpm, danceability_%:speechiness_%` range select), scale them.

In [3]:
FEATURE_COLS = [
    'bpm', 'danceability_%', 'valence_%', 'energy_%',
    'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%'
]

X = data[FEATURE_COLS].copy()
X_scaled = StandardScaler().fit_transform(X)

## Compute Euclidean Norms

For each song, compute the Euclidean norm of its scaled feature vector — a single number used as a coarse similarity handle.

In [4]:
data['euclidean_norm'] = np.linalg.norm(X_scaled, axis=1)

## Song Lookup Helper

Case-insensitive exact match, with fuzzy "did you mean" suggestions (`difflib.get_close_matches`, Python's standard-library equivalent of R's `agrep`) if there's no exact match.

In [5]:
def check_song(song_name, titles):
    song_name_lower = song_name.lower().strip()
    titles_lower = titles.str.lower().str.strip()

    matches = titles_lower[titles_lower == song_name_lower]
    if len(matches) > 0:
        return matches.index[0]

    close = difflib.get_close_matches(song_name_lower, titles_lower.tolist(), n=5, cutoff=0.6)
    if close:
        print("Did you mean one of these songs?")
        for c in close:
            print(" -", c)
    else:
        print("Song not found in the dataset.")
    return None

# check_song("mastermind", data['track_name'])

## Recommender Function

Recommends the 5 nearest songs by Euclidean-norm distance, re-ordering to prioritize a shared artist if one exists among the candidates.

In [6]:
def recommender(song_name):
    idx = check_song(song_name, data['track_name'])
    if idx is None:
        return None

    dist = (data['euclidean_norm'] - data['euclidean_norm'].loc[idx]).abs()
    ranked = data.assign(dist=dist).sort_values('dist')

    # exclude the song itself (dist == 0, first row), take next 5 closest
    candidates = ranked.iloc[1:6]

    query_artists = set(a.strip().lower() for a in str(data['artist(s)_name'].loc[idx]).split(','))

    def shares_artist(artists_str):
        candidate_artists = set(a.strip().lower() for a in str(artists_str).split(','))
        return len(query_artists & candidate_artists) > 0

    shares_mask = candidates['artist(s)_name'].apply(shares_artist)
    boosted = candidates[shares_mask]
    rest = candidates[~shares_mask]

    output = pd.concat([boosted, rest])['track_name'].drop_duplicates().tolist()
    return output

## Example Usage

In [8]:
recommender("cruel summer")

['Yandel 150', 'Nos Comemos (feat. Ozuna)', 'Ghost', 'Born Singer', 'Sigue']